# 01 — Prise en main

Ce que `better-calendar` remplace, en une ligne : la boucle `while d.weekday() > 4: d += timedelta(days=1)`
que tout le monde a réécrite trois fois, et qui est fausse dès qu'un jour férié entre en jeu.

Le modèle mental tient en une phrase : **un calendrier est un tableau `int64` trié de jours ouvrés**
sur un horizon borné. Appartenance, décalage, comptage, algèbre d'ensembles — tout se ramène à un
`searchsorted` sur ce tableau.

Au programme de ce notebook :

- les trois fonctions qu'on utilise 90 % du temps ;
- la transparence de type (ce qui entre ressort dans le même type) ;
- les conventions de roll ISDA ;
- les bornes, et pourquoi elles lèvent ;
- la vectorisation, mesurée.

In [1]:
from datetime import date, datetime, timedelta

import numpy as np
import pandas as pd

import better_calendar as bcal

bcal.__version__

'1.0.0'

## 1. Les trois fonctions du quotidien

`adjust` normalise, `offset` décale, `count` compte. Toutes les trois prennent un calendrier
en mot-clé `cal=` : un objet `Calendar`, un identifiant, ou `None` pour le calendrier
`weekday` (lundi–vendredi, sans férié).

In [2]:
# 2026-08-01 est un samedi.
print("adjust  :", bcal.adjust("2026-08-01"))                    # jour ouvré suivant
print("offset  :", bcal.offset("2026-07-31", 5))                 # 5 jours ouvrés plus tard
print("count   :", bcal.count("2026-07-27", "2026-08-01"))       # intervalle semi-ouvert

adjust  : 2026-08-03
offset  : 2026-08-07
count   : 5


Avec un vrai calendrier de place, la différence saute aux yeux — le 3 juillet 2026 est férié
au NYSE (Independence Day observé le vendredi) :

In [3]:
for name in ("weekday", "XNYS", "fin:TARGET2"):
    print(f"{name:14s} 2026-07-02 + 1 jour ouvré -> {bcal.offset('2026-07-02', 1, cal=name)}")

weekday        2026-07-02 + 1 jour ouvré -> 2026-07-03
XNYS           2026-07-02 + 1 jour ouvré -> 2026-07-06
fin:TARGET2    2026-07-02 + 1 jour ouvré -> 2026-07-03


## 2. Transparence de type

Ce qui entre ressort dans le même type. Pas de `pd.Timestamp` surprise au milieu d'un
pipeline de `date`, pas de chaîne transformée en objet.

In [4]:
entrees = [
    date(2026, 7, 31),
    datetime(2026, 7, 31, 9, 30),
    pd.Timestamp("2026-07-31 09:30"),
    np.datetime64("2026-07-31"),
    "2026-07-31",
    "20260731",
    20260731,
]

lignes = []
for valeur in entrees:
    resultat = bcal.offset(valeur, 1)
    lignes.append(
        {
            "type entrée": type(valeur).__name__,
            "valeur": repr(valeur),
            "résultat": repr(resultat),
            "type sortie": type(resultat).__name__,
        }
    )
pd.DataFrame(lignes)

,type entrée,valeur,résultat,type sortie
0,date,"datetime.date(2026, 7, 31)","datetime.date(2026, 8, 3)",date
1,datetime,"datetime.datetime(2026, 7, 31, 9, 30)","datetime.datetime(2026, 8, 3, 9, 30)",datetime
2,Timestamp,Timestamp('2026-07-31 09:30:00'),Timestamp('2026-08-03 09:30:00'),Timestamp
3,datetime64,np.datetime64('2026-07-31'),np.datetime64('2026-08-03'),datetime64
4,str,'2026-07-31','2026-08-03',str
5,str,'20260731','20260803',str
6,int,20260731,20260803,int


Deux choses à noter :

- le `datetime` à 09:30 ressort **à 09:30** : un décalage ne touche que la partie date ;
- l'entier est lu comme `yyyymmdd`, jamais comme un timestamp Unix.

Le parsing est volontairement strict. Les formats ambigus sont refusés plutôt que devinés :

In [5]:
for texte in ("31/07/2026", "07/31/2026", "Jul 31 2026"):
    try:
        bcal.to_date(texte)
    except bcal.BetterCalendarError as exc:
        print(f"{texte!r:16s} -> {str(exc)[:95]}…")

'31/07/2026'     -> Cannot parse date string '31/07/2026'. Accepted formats are ISO-8601 ('2026-07-31', '2026-07-31…
'07/31/2026'     -> Cannot parse date string '07/31/2026'. Accepted formats are ISO-8601 ('2026-07-31', '2026-07-31…
'Jul 31 2026'    -> Cannot parse date string 'Jul 31 2026'. Accepted formats are ISO-8601 ('2026-07-31', '2026-07-3…


In [6]:
# Un timestamp Unix passé par erreur ne devient pas silencieusement une date.
try:
    bcal.to_date(1785456000)
except bcal.BetterCalendarError as exc:
    print(exc)

Cannot interpret 1785456000 as a date. Ints are read as yyyymmdd (for example 20260731); this guards against Unix timestamps being passed by accident. Pass a datetime.date or an ISO-8601 string instead.


## 3. Conventions de roll

Sept conventions, accessibles par leur nom complet ou leur alias ISDA court
(`"F"`, `"MF"`, `"P"`, `"MP"`, `"N"`), insensible à la casse.

Le tableau ci-dessous les applique à deux dates piégeuses : un samedi ordinaire, et le
**dimanche 31 mai 2026** — dernier jour de son mois, ce qui fait toute la différence entre
`FOLLOWING` et `MODIFIED_FOLLOWING`.

In [7]:
dates = ["2026-08-01", "2026-05-31", "2026-02-01"]
tableau = {}
for roll in bcal.Roll:
    ligne = {}
    for jour in dates:
        try:
            ligne[jour] = bcal.adjust(jour, roll)
        except bcal.BetterCalendarError as exc:
            ligne[jour] = f"<{type(exc).__name__}>"
    tableau[roll.value] = ligne

resultat = pd.DataFrame(tableau).T
resultat.columns = [f"{c} ({date.fromisoformat(c).strftime('%a')})" for c in resultat.columns]
resultat

,2026-08-01 (Sat),2026-05-31 (Sun),2026-02-01 (Sun)
none,2026-08-01,2026-05-31,2026-02-01
following,2026-08-03,2026-06-01,2026-02-02
preceding,2026-07-31,2026-05-29,2026-01-30
modified_following,2026-08-03,2026-05-29,2026-02-02
modified_preceding,2026-08-03,2026-05-29,2026-02-02
nearest,2026-07-31,2026-06-01,2026-02-02
raise,<NotABusinessDayError>,<NotABusinessDayError>,<NotABusinessDayError>


Lecture des deux colonnes intéressantes :

- **31 mai** (dimanche, fin de mois) : `following` sort de mai vers le 1er juin, donc
  `modified_following` fait demi-tour sur le vendredi 29.
- **1er février** (dimanche, début de mois) : `preceding` sort de février vers le 30 janvier,
  donc `modified_preceding` repart en avant sur le lundi 2.

C'est exactement pour ça que les variantes « modified » existent, et c'est là que les
implémentations maison se trompent.

In [8]:
# Roll.RAISE refuse au lieu d'ajuster — utile pour valider une saisie.
try:
    bcal.adjust("2026-08-01", bcal.Roll.RAISE)
except bcal.NotABusinessDayError as exc:
    print(exc)

2026-08-01 is not a business day in calendar 'weekday' and roll=Roll.RAISE forbids adjusting it. Pass a different roll convention (for example Roll.MODIFIED_FOLLOWING) to move it to a nearby business day.


## 4. Comptage et intervalles

L'intervalle semi-ouvert `[début, fin)` est la valeur par défaut **partout**. Toute autre
convention doit être demandée explicitement.

In [9]:
debut, fin = "2026-07-27", "2026-07-31"   # lundi -> vendredi
pd.DataFrame(
    [
        {"closed": c, "count": bcal.count(debut, fin, closed=c)}
        for c in ("left", "right", "both", "neither")
    ]
).set_index("closed")

,count
closed,
left,4
right,4
both,5
neither,3


Le comptage est **signé**, ce qui rend la propriété `count(d, offset(d, n)) == n` vraie
aussi pour `n` négatif :

In [10]:
depart = "2026-07-31"
for n in (-10, -1, 0, 1, 10):
    arrivee = bcal.offset(depart, n)
    assert bcal.count(depart, arrivee) == n
    print(f"offset({depart}, {n:>3}) = {arrivee}   count -> {bcal.count(depart, arrivee):>3}")

offset(2026-07-31, -10) = 2026-07-17   count -> -10
offset(2026-07-31,  -1) = 2026-07-30   count ->  -1
offset(2026-07-31,   0) = 2026-07-31   count ->   0
offset(2026-07-31,   1) = 2026-08-03   count ->   1
offset(2026-07-31,  10) = 2026-08-14   count ->  10


`DateRange` est le petit objet valeur qui va avec :

In [11]:
periode = bcal.DateRange("2026-07-27", "2026-08-03")
print("longueur          :", len(periode))
print("3 août dedans ?   :", date(2026, 8, 3) in periode)      # semi-ouvert : non
print("jours ouvrés      :", len(periode.business_days()))
print("jours 24/7        :", len(periode.business_days("crypto:24x7")))

trimestre = bcal.DateRange("2026-01-01", "2026-04-01")
print("\ndécoupage mensuel :", [str(p.start) for p in trimestre.split("M")])
print("les morceaux pavent sans trou :",
      sum(len(p) for p in trimestre.split("M")) == len(trimestre))

longueur          : 7
3 août dedans ?   : False
jours ouvrés      : 5
jours 24/7        : 7

découpage mensuel : ['2026-01-01', '2026-02-01', '2026-03-01']
les morceaux pavent sans trou : True


## 5. Bornes : ne jamais extrapoler

Chaque calendrier a un horizon fini et explicite. En sortir lève une erreur qui dit
**où** sont les bornes — jamais une réponse inventée.

In [12]:
# L'horizon par défaut est piloté par une seule constante, jamais par un littéral épars.
print("horizon par défaut :", bcal.MIN_YEAR, "->", bcal.MAX_YEAR)
print("bornes weekday     :", bcal.get("weekday").bounds)

horizon par défaut : 1970 -> 2100
bornes weekday     : (datetime.date(1970, 1, 1), datetime.date(2100, 12, 31))


In [13]:
tokyo = bcal.get("XTKS")
print("bornes XTKS :", tokyo.bounds)

try:
    tokyo.is_bday("1996-12-31")
except bcal.OutOfBoundsError as exc:
    print("\n" + str(exc))

bornes XTKS : (datetime.date(1997, 1, 1), datetime.date(2100, 12, 31))

1996-12-31 is outside the bounds of calendar 'XTKS' (1997-01-01 to 2100-12-31, inclusive). Rebuild the calendar with wider `bounds`, or raise MAX_YEAR in better_calendar.core.epoch.


Ces bornes ne sont pas décoratives : elles reflètent ce que la source amont sait
réellement répondre. `exchange-calendars` refuse d'évaluer Tokyo avant 1997, Hong Kong
après 2049, et les calendriers lunaires de QuantLib s'arrêtent là où leurs tables
s'arrêtent.

In [14]:
pd.DataFrame(
    [
        {"calendrier": n, "début": bcal.get(n).bounds[0], "fin": bcal.get(n).bounds[1]}
        for n in ("weekday", "XNYS", "XTKS", "XHKG", "ql:Israel.TASE", "ql:China.SSE")
    ]
).set_index("calendrier")

,début,fin
calendrier,,
weekday,1970-01-01,2100-12-31
XNYS,1970-01-01,2100-12-31
XTKS,1997-01-01,2100-12-31
XHKG,1970-01-01,2049-12-31
ql:Israel.TASE,1970-01-01,2025-12-31
ql:China.SSE,1970-01-01,2026-12-31


## 6. Vectorisation

Toute fonction scalaire accepte aussi un tableau. Le gain n'est pas dans l'algorithme —
le chemin scalaire est déjà en `searchsorted`, donc O(log n) — mais dans le fait de ne
pas repayer l'itération Python et la conversion de type à chaque élément.

In [15]:
import time

jours = pd.date_range("2020-01-01", "2026-12-31", freq="D")
calendrier = bcal.get("XNYS")

debut_v = time.perf_counter()
vectorise = calendrier.offset(jours, 5)
temps_v = time.perf_counter() - debut_v

debut_b = time.perf_counter()
boucle = [calendrier.offset(j, 5) for j in jours[:2000]]
temps_b = (time.perf_counter() - debut_b) * len(jours) / 2000

print(f"{len(jours)} dates")
print(f"  vectorisé   : {temps_v * 1000:8.1f} ms")
print(f"  un par un   : {temps_b * 1000:8.1f} ms  (extrapolé)")
print(f"  rapport     : {temps_b / temps_v:8.0f}×")

2557 dates
  vectorisé   :      1.6 ms
  un par un   :     34.9 ms  (extrapolé)
  rapport     :       22×


## Récapitulatif

| Fonction | Rôle |
|---|---|
| `adjust(d, roll, cal=)` | ramener une date sur un jour ouvré |
| `offset(d, n, cal=)` | décaler de `n` jours ouvrés |
| `count(a, b, cal=)` | compter les jours ouvrés, signé |
| `is_bday`, `next_bday`, `prev_bday` | appartenance et voisins |
| `to_date`, `to_datetime`, `to_timestamp` | conversions strictes |
| `DateRange` | intervalle, itération, découpage |
| `Roll` | les sept conventions ISDA |

**Suite :** [02 — Calendriers et algèbre](02-calendriers-et-algebre.ipynb)